EDA del archivo credits.csv

In [1]:
import pandas as pd

In [2]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

In [3]:
credits_df = pd.read_csv('../src/credits.csv')
credits_df.head(1)

FileNotFoundError: [Errno 2] No such file or directory: '../src/credits.csv'

In [6]:
# Invented credits_df info
# credits_data = {"id": [1, 2, 3, 4, 5],
#                 "cast_names": ["John Connor, Jessica Lange", "Pepe Marrone", "Ursula Andrews, Oriana Fallaci", "Andrea", "Chloe"]}
# credits_df = pd.DataFrame(credits_data)
# credits_df

,id,cast_names
0,1,"John Connor, Jessica Lange"
1,2,Pepe Marrone
2,3,"Ursula Andrews, Oriana Fallaci"
3,4,Andrea
4,5,Chloe


In [4]:
credits_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45476 entries, 0 to 45475
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   cast    45476 non-null  object
 1   crew    45476 non-null  object
 2   id      45476 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.0+ MB


In [ ]:
import ast
ast.literal_eval(credits_df.loc[45474,'cast'])

In [ ]:
import etl_functions as etlf
credits_df['cast_names'] = credits_df['cast'].apply(etlf.obtener_valores, args=('name',))
credits_df['cast_names']


In [9]:
m_cast_df = credits_df[['id', 'cast_names']]
m_cast_df

,id,cast_names
0,1,"John Connor, Jessica Lange"
1,2,Pepe Marrone
2,3,"Ursula Andrews, Oriana Fallaci"
3,4,Andrea
4,5,Chloe


In [10]:
df_expanded = m_cast_df['cast_names'].str.split(', ', expand=True).stack().reset_index(level=1, drop=True).to_frame('actor_actress_name')
df_expanded

,actor_actress_name
0,John Connor
0,Jessica Lange
1,Pepe Marrone
2,Ursula Andrews
2,Oriana Fallaci
3,Andrea
4,Chloe


In [ ]:
m_cast_df = m_cast_df.join(df_expanded)
m_cast_df.drop(columns=['cast_names'], inplace=True)
m_cast_df

In [ ]:
credits_df['crew_names'] = credits_df['crew'].apply(etlf.obtener_valores, args=('name',))
credits_df['job_names'] = credits_df['crew'].apply(etlf.obtener_valores, args=('job',))
credits_df[['job_names', 'crew_names']]

m_crew_job_df = credits_df[['id', 'job_names']]
df_expanded = m_crew_job_df['job_names'].str.split(', ', expand=True).stack().reset_index(level=1, drop=True).to_frame('job_name')
m_crew_job_df = m_crew_job_df.join(df_expanded)
m_crew_job_df.drop(columns=['job_names'], inplace=True)

m_crew_name_df = credits_df[['id', 'crew_names']]
df_expanded = m_crew_name_df['crew_names'].str.split(', ', expand=True).stack().reset_index(level=1, drop=True).to_frame('crew_name')
m_crew_name_df = m_crew_name_df.join(df_expanded)
m_crew_name_df.drop(columns=['crew_names'], inplace=True)

In [ ]:
m_crew_job_df

In [ ]:
m_crew_name_df

In [5]:
import etl_flow as etlflow

In [7]:
%env DIRECTORIO_RAIZ=/com.docker.devenvironments.code

env: DIRECTORIO_RAIZ=/com.docker.devenvironments.code


In [8]:
d = etlflow.obtener_credits_dataframes()

In [9]:
d

{'m_cast_df':            id actor_actress_name
 0         862          Tom Hanks
 0         862          Tim Allen
 0         862        Don Rickles
 0         862         Jim Varney
 0         862      Wallace Shawn
 ...       ...                ...
 45474  227506  Nathalie Lissenko
 45474  227506       Pavel Pavlov
 45474  227506  Aleksandr Chabrov
 45474  227506        Vera Orlova
 45475  461257                   
 
 [565219 rows x 2 columns],
 'm_crew_job_and_name_df':            id       job              name
 0         862  Director     John Lasseter
 1        8844  Director      Joe Johnston
 2       15602  Director     Howard Deutch
 3       31357  Director   Forest Whitaker
 4       11862  Director     Charles Shyer
 ...       ...       ...               ...
 45471  439050  Director  Hamid Nematollah
 45472  111109  Director          Lav Diaz
 45473   67758  Director    Mark L. Lester
 45474  227506  Director  Yakov Protazanov
 45475  461257  Director     Daisy Asquith
 
 [490

             m_cast_df : 565219 rows de actores y actrices
m_crew_job_and_name_df : 49048 rows de directores

In [12]:
m_cast_df = d['m_cast_df']
m_cast_df.to_csv('m_cast_df.tsv', sep='\t', index=False)

In [ ]:
m_crew_job_and_name_df = d['m_crew_job_and_name_df']
m_crew_job_and_name_df.to_csv('m_crew_job_and_name_df.tsv', sep='\t', index=False)

In [7]:
%env DIRECTORIO_RAIZ=/com.docker.devenvironments.code

env: DIRECTORIO_RAIZ=/com.docker.devenvironments.code


In [ ]:
import os
import pandas as pd
import etl_functions as etlf

# Obtengo el directorio raiz desde la variable de entorno DIRECTORIO_RAIZ
dir_raiz = os.getenv("DIRECTORIO_RAIZ")

# Credits
# =======
archivo = os.path.join(dir_raiz, 'src/credits.csv_no')
credits_df = pd.read_csv(archivo)

# Tratamiento columna cast. Genero un nuevo dataframe con los actores/actrices expandidos y el id para hacer join
credits_df['cast_names'] = credits_df['cast'].apply(etlf.obtener_valores, args=('name',))
credits_df.drop(columns=['cast'], inplace=True)
m_cast_df = credits_df[['id', 'cast_names']]
df_expanded = m_cast_df['cast_names'].str.split(', ', expand=True).stack().reset_index(level=1, drop=True).to_frame('actor_actress_name')
m_cast_df = m_cast_df.join(df_expanded)
del df_expanded
m_cast_df.drop(columns=['cast_names'], inplace=True)

# Tratamiento columna crew. Genero un nuevo dataframe con id (Para hacer join), nombre del trabajo y nombre de la persona
credits_df['jobs_and_names'] = credits_df['crew'].apply(etlf.obtener_cargo_y_nombre)
credits_df.drop(columns=['crew'], inplace=True)
m_crew_job_and_name_df = credits_df[['id', 'jobs_and_names']]
df_expanded = m_crew_job_and_name_df['jobs_and_names'].str.split(', ', expand=True).stack().reset_index(level=1, drop=True).to_frame('job_and_name')
m_crew_job_and_name_df = m_crew_job_and_name_df.join(df_expanded)
del df_expanded
m_crew_job_and_name_df.drop(columns=['jobs_and_names'], inplace=True)
m_crew_job_and_name_df[['job', 'name']] = m_crew_job_and_name_df['job_and_name'].str.split(':-:', expand=True)
m_crew_job_and_name_df.drop(columns=['job_and_name'], inplace=True)
# Conservo unicamente los directores para hacer el dataframe mas chico
m_crew_job_and_name_df = m_crew_job_and_name_df[m_crew_job_and_name_df['job'] == 'Director']

# Libero memoria (Requerido por Render)
del credits_df

# RESUMEN
# =======
# Los credits dataframes del modelo son
#              m_cast_df : Es el dataframe de actores y actrices en cada pelicula (Clave id)
# m_crew_job_and_name_df : Es el dataframe de trabajos del reparto en cada pelicula (Clave id)
